In [1]:
#!/usr/bin/env python3
"""
Script para extrair rotas de ônibus de Londrina
Fonte: https://bus2.info/2you/#/2fvn7
"""

import requests
import json
import polyline
import geopandas as gpd
from shapely.geometry import LineString, Point
import pandas as pd
from time import sleep
from tqdm import tqdm
import sys

# Constantes
BASE_URL = "https://mobilibus.com/api"
PROJECT_ID = 83
OUTPUT_DIR = "."

def get_all_routes():
    """Busca todas as linhas disponíveis"""
    url = f"{BASE_URL}/routes?origin=web&project_id={PROJECT_ID}"
    print(f"Buscando: {url}")
    
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()
        
        # Validar resposta
        if not isinstance(data, list):
            print(f"ERRO: API retornou tipo inesperado: {type(data)}")
            print(f"Primeiros 500 caracteres: {str(data)[:500]}")
            return []
        
        print(f"✓ API retornou {len(data)} linhas")
        return data
        
    except requests.exceptions.RequestException as e:
        print(f"ERRO ao conectar com a API: {e}")
        return []
    except json.JSONDecodeError as e:
        print(f"ERRO ao decodificar JSON: {e}")
        return []
    except Exception as e:
        print(f"ERRO inesperado: {e}")
        return []

def get_route_timetable(route_id):
    """Busca os detalhes de horários e trips de uma rota"""
    url = f"{BASE_URL}/timetable?origin=web&v=2&project_id={PROJECT_ID}&route_id={route_id}"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"  └─ Erro ao buscar timetable: {e}")
        return {}

def get_trip_details(trip_id):
    """Busca os detalhes de uma viagem específica (inclui shape e stops)"""
    url = f"{BASE_URL}/trip-details?origin=web&v=2&trip_id={trip_id}"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"    └─ Erro ao buscar trip_details: {e}")
        return {}

def decode_polyline_shape(encoded_shape):
    """Decodifica uma string polyline em coordenadas lat/lng"""
    try:
        coordinates = polyline.decode(encoded_shape)
        # Converte de (lat, lng) para (lng, lat) para GeoJSON
        return [(lng, lat) for lat, lng in coordinates]
    except Exception as e:
        print(f"    └─ Erro ao decodificar polyline: {e}")
        return []

def extract_all_routes():
    """Função principal que extrai todas as rotas"""
    
    print("\n" + "="*60)
    print("EXTRAÇÃO DE ROTAS DE ÔNIBUS - LONDRINA")
    print("="*60 + "\n")
    
    # Busca lista de linhas
    routes = get_all_routes()
    
    if not routes:
        print("\nNENHUMA LINHA ENCONTRADA!")
        print("Verifique sua conexão com a internet.")
        return [], []
    
    print(f"\nTotal de linhas encontradas: {len(routes)}\n")
    
    all_routes_data = []
    all_stops_data = []
    
    # Estatísticas
    stats = {
        'linhas_processadas': 0,
        'linhas_com_erro': 0,
        'trips_processadas': 0,
        'rotas_extraidas': 0,
        'paradas_extraidas': 0
    }
    
    # Processa cada linha
    for route in tqdm(routes, desc="Processando linhas", unit="linha"):
        
        # Validar estrutura
        if not isinstance(route, dict):
            print(f"\n⚠ Linha com formato inválido: {type(route)}")
            stats['linhas_com_erro'] += 1
            continue
        
        required_fields = ['routeId', 'shortName', 'longName', 'color', 'agencyId']
        if not all(field in route for field in required_fields):
            print(f"\n⚠ Linha sem campos necessários: {route.get('shortName', '?')}")
            stats['linhas_com_erro'] += 1
            continue
        
        route_id = route['routeId']
        route_name = f"{route['shortName']} - {route['longName']}"
        
        try:
            # Busca informações de timetable
            timetable = get_route_timetable(route_id)
            
            if not timetable or 'timetable' not in timetable:
                tqdm.write(f"  {route_name}: Sem dados de timetable")
                stats['linhas_com_erro'] += 1
                continue
            
            if 'trips' not in timetable['timetable']:
                tqdm.write(f"  {route_name}: Sem trips disponíveis")
                stats['linhas_com_erro'] += 1
                continue
            
            trips = timetable['timetable']['trips']
            
            # Processa cada trip
            for trip in trips:
                if not isinstance(trip, dict) or 'tripId' not in trip:
                    continue
                
                trip_id = trip['tripId']
                trip_name = trip.get('tripDesc', 'Sem nome')
                
                try:
                    # Busca detalhes da trip
                    trip_details = get_trip_details(trip_id)
                    
                    if not trip_details:
                        continue
                    
                    stats['trips_processadas'] += 1
                    
                    # Processa shape (geometria da rota)
                    if 'shape' in trip_details and trip_details['shape']:
                        coordinates = decode_polyline_shape(trip_details['shape'])
                        
                        if coordinates and len(coordinates) >= 2:
                            all_routes_data.append({
                                'route_id': route_id,
                                'route_short_name': route['shortName'],
                                'route_long_name': route['longName'],
                                'trip_id': trip_id,
                                'trip_name': trip_name,
                                'route_color': route['color'],
                                'agency_id': route['agencyId'],
                                'geometry': LineString(coordinates)
                            })
                            stats['rotas_extraidas'] += 1
                    
                    # Processa stops (paradas)
                    if 'stops' in trip_details and isinstance(trip_details['stops'], list):
                        for stop in trip_details['stops']:
                            if not isinstance(stop, dict):
                                continue
                            
                            required_stop_fields = ['stopId', 'name', 'lat', 'lng']
                            if not all(field in stop for field in required_stop_fields):
                                continue
                            
                            all_stops_data.append({
                                'route_id': route_id,
                                'route_short_name': route['shortName'],
                                'route_long_name': route['longName'],
                                'trip_id': trip_id,
                                'trip_name': trip_name,
                                'stop_id': stop['stopId'],
                                'stop_name': stop['name'],
                                'geometry': Point(stop['lng'], stop['lat'])
                            })
                            stats['paradas_extraidas'] += 1
                    
                    sleep(0.05)  # Pausa breve
                    
                except Exception as e:
                    tqdm.write(f"    Erro na trip {trip_id}: {str(e)[:50]}")
                    continue
            
            stats['linhas_processadas'] += 1
            sleep(0.1)  # Pausa entre linhas
            
        except KeyboardInterrupt:
            print("\n\n⚠ Interrompido pelo usuário!")
            break
        except Exception as e:
            tqdm.write(f"  Erro na linha {route_name}: {str(e)[:50]}")
            stats['linhas_com_erro'] += 1
            continue
    
    # Exibir estatísticas
    print("\n" + "="*60)
    print("ESTATÍSTICAS DA EXTRAÇÃO")
    print("="*60)
    print(f"Linhas processadas com sucesso: {stats['linhas_processadas']}")
    print(f"Linhas com erro: {stats['linhas_com_erro']}")
    print(f"Trips processadas: {stats['trips_processadas']}")
    print(f"Rotas extraídas: {stats['rotas_extraidas']}")
    print(f"Paradas extraídas: {stats['paradas_extraidas']}")
    print("="*60 + "\n")
    
    return all_routes_data, all_stops_data

def save_to_files(routes_data, stops_data, output_dir):
    """Salva os dados em múltiplos formatos"""
    
    if not routes_data and not stops_data:
        print("⚠ Nenhum dado para salvar!")
        return
    
    print("Salvando dados...\n")
    
    # Salva rotas
    if routes_data:
        routes_gdf = gpd.GeoDataFrame(routes_data, crs="EPSG:4326")
        
        routes_gdf.to_file(f"{output_dir}/bus_routes.geojson", driver="GeoJSON")
        print(f"✓ bus_routes.geojson - {len(routes_data)} rotas")
        
        routes_gdf.to_file(f"{output_dir}/bus_routes.shp")
        print(f"✓ bus_routes.shp - {len(routes_data)} rotas")
        
        routes_gdf.to_file(f"{output_dir}/bus_routes.gpkg", driver="GPKG")
        print(f"✓ bus_routes.gpkg - {len(routes_data)} rotas")
    
    # Salva paradas
    if stops_data:
        stops_gdf = gpd.GeoDataFrame(stops_data, crs="EPSG:4326")
        
        stops_gdf.to_file(f"{output_dir}/bus_stops.geojson", driver="GeoJSON")
        print(f"✓ bus_stops.geojson - {len(stops_data)} paradas")
        
        stops_gdf.to_file(f"{output_dir}/bus_stops.shp")
        print(f"✓ bus_stops.shp - {len(stops_data)} paradas")
        
        stops_gdf.to_file(f"{output_dir}/bus_stops.gpkg", driver="GPKG")
        print(f"✓ bus_stops.gpkg - {len(stops_data)} paradas")
    
    # Salva metadados
    metadata = {
        'total_routes': len(set([r['route_id'] for r in routes_data])) if routes_data else 0,
        'total_trips': len(routes_data),
        'total_stops': len(stops_data),
        'crs': 'EPSG:4326',
        'source': 'https://bus2.info/2you/#/2fvn7',
        'project_id': PROJECT_ID
    }
    
    with open(f"{output_dir}/metadata.json", 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print(f"✓ metadata.json")
    
    # Cria resumo
    if routes_data:
        routes_summary = pd.DataFrame([
            {
                'route_id': r['route_id'],
                'short_name': r['route_short_name'],
                'long_name': r['route_long_name'],
                'color': r['route_color']
            }
            for r in routes_data
        ]).drop_duplicates(subset=['route_id'])
        
        routes_summary.to_csv(f"{output_dir}/routes_summary.csv", index=False)
        print(f"✓ routes_summary.csv - {len(routes_summary)} linhas únicas")
    
    print("\n" + "="*60)
    print("ARQUIVOS SALVOS COM SUCESSO!")
    print("="*60)

if __name__ == "__main__":
    try:
        # Extrai dados
        routes_data, stops_data = extract_all_routes()
        
        # Salva resultados
        if routes_data or stops_data:
            save_to_files(routes_data, stops_data, OUTPUT_DIR)
        else:
            print("\n⚠ Nenhum dado foi extraído!")
            print("Verifique:")
            print("  1. Conexão com a internet")
            print("  2. Acesso ao site bus2.info")
            print("  3. Se a API não mudou")
            sys.exit(1)
            
    except KeyboardInterrupt:
        print("\n\n⚠ Execução interrompida pelo usuário!")
        sys.exit(1)
    except Exception as e:
        print(f"\n❌ ERRO FATAL: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)


EXTRAÇÃO DE ROTAS DE ÔNIBUS - LONDRINA

Buscando: https://mobilibus.com/api/routes?origin=web&project_id=83
✓ API retornou 140 linhas

Total de linhas encontradas: 140



Processando linhas: 100%|██████████| 140/140 [07:07<00:00,  3.05s/linha]
/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_30652/480458961.py:250: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  routes_gdf.to_file(f"{output_dir}/bus_routes.shp")



ESTATÍSTICAS DA EXTRAÇÃO
Linhas processadas com sucesso: 140
Linhas com erro: 0
Trips processadas: 636
Rotas extraídas: 636
Paradas extraídas: 14632

Salvando dados...

✓ bus_routes.geojson - 636 rotas


/opt/homebrew/Caskroom/miniconda/base/envs/gee_env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'route_short_name' to 'route_shor'
  ogr_write(
/opt/homebrew/Caskroom/miniconda/base/envs/gee_env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'route_long_name' to 'route_long'
  ogr_write(
/opt/homebrew/Caskroom/miniconda/base/envs/gee_env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'route_color' to 'route_colo'
  ogr_write(
/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_30652/480458961.py:263: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  stops_gdf.to_file(f"{output_dir}/bus_stops.shp")
/opt/homebrew/Caskroom/miniconda/base/envs/gee_env/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'route_short_name' to 'route_sho

✓ bus_routes.shp - 636 rotas
✓ bus_routes.gpkg - 636 rotas
✓ bus_stops.geojson - 14632 paradas
✓ bus_stops.shp - 14632 paradas
✓ bus_stops.gpkg - 14632 paradas
✓ metadata.json
✓ routes_summary.csv - 140 linhas únicas

ARQUIVOS SALVOS COM SUCESSO!


In [2]:
#!/usr/bin/env python3
"""
Gera mapa HTML interativo com todas as rotas de ônibus
"""

import geopandas as gpd
import folium
from folium import plugins
import json

def create_interactive_map(routes_file, stops_file, output_html='bus_routes_map.html'):
    """
    Cria mapa interativo HTML com todas as rotas e paradas
    
    Parameters:
    -----------
    routes_file : str
        Caminho para o arquivo de rotas (GeoJSON, Shapefile, etc.)
    stops_file : str
        Caminho para o arquivo de paradas
    output_html : str
        Nome do arquivo HTML de saída
    """
    
    print("Carregando dados...")
    
    # Carregar dados
    routes = gpd.read_file(routes_file)
    stops = gpd.read_file(stops_file)
    
    print(f"✓ {len(routes)} rotas carregadas")
    print(f"✓ {len(stops)} paradas carregadas")
    
    # Converter para EPSG:4326 se necessário
    if routes.crs != 'EPSG:4326':
        routes = routes.to_crs('EPSG:4326')
    if stops.crs != 'EPSG:4326':
        stops = stops.to_crs('EPSG:4326')
    
    # Calcular centro do mapa
    bounds = routes.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    
    print(f"\nCriando mapa centrado em: {center_lat:.4f}, {center_lon:.4f}")
    
    # Criar mapa base
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='OpenStreetMap',
        control_scale=True
    )
    
    # Adicionar diferentes tiles
    folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m)
    folium.TileLayer('CartoDB dark_matter', name='CartoDB Dark').add_to(m)
    
    # Criar grupos de camadas por linha
    unique_routes = routes[['route_id', 'route_short_name', 'route_long_name', 'route_color']].drop_duplicates()
    
    print(f"\nAdicionando {len(unique_routes)} linhas ao mapa...")
    
    # Dicionário para armazenar feature groups
    route_groups = {}
    
    for idx, route_info in unique_routes.iterrows():
        route_id = route_info['route_id']
        route_name = f"{route_info['route_short_name']} - {route_info['route_long_name']}"
        route_color = route_info['route_color']
        
        # Criar grupo de features para esta linha
        feature_group = folium.FeatureGroup(name=route_name, show=False)
        
        # Adicionar todas as rotas desta linha
        route_trips = routes[routes['route_id'] == route_id]
        
        for trip_idx, trip in route_trips.iterrows():
            # Extrair coordenadas
            coords = [(coord[1], coord[0]) for coord in trip.geometry.coords]
            
            # Adicionar polilinha
            folium.PolyLine(
                coords,
                color=route_color,
                weight=3,
                opacity=0.7,
                popup=folium.Popup(
                    f"<b>{route_name}</b><br>{trip['trip_name']}",
                    max_width=200
                ),
                tooltip=route_name
            ).add_to(feature_group)
        
        # Adicionar paradas desta linha
        route_stops = stops[stops['route_id'] == route_id]
        
        for stop_idx, stop in route_stops.iterrows():
            folium.CircleMarker(
                location=[stop.geometry.y, stop.geometry.x],
                radius=3,
                color=route_color,
                fill=True,
                fillColor=route_color,
                fillOpacity=0.7,
                popup=folium.Popup(
                    f"<b>{stop['stop_name']}</b><br>Linha: {route_name}",
                    max_width=250
                ),
                tooltip=stop['stop_name']
            ).add_to(feature_group)
        
        feature_group.add_to(m)
        route_groups[route_id] = feature_group
    
    # Adicionar layer control
    folium.LayerControl(collapsed=False).add_to(m)
    
    # Adicionar plugins úteis
    plugins.Fullscreen().add_to(m)
    plugins.MeasureControl().add_to(m)
    plugins.LocateControl().add_to(m)
    
    # Adicionar minimap
    minimap = plugins.MiniMap(toggle_display=True)
    m.add_child(minimap)
    
    # Adicionar legenda
    legend_html = '''
    <div style="position: fixed; 
                bottom: 50px; left: 50px; width: 300px; height: auto; 
                background-color: white; z-index:9999; font-size:14px;
                border:2px solid grey; border-radius: 5px; padding: 10px">
    <h4 style="margin-top:0">Rotas de Ônibus - Londrina</h4>
    <p><b>Total de linhas:</b> ''' + str(len(unique_routes)) + '''</p>
    <p><b>Total de rotas:</b> ''' + str(len(routes)) + '''</p>
    <p><b>Total de paradas:</b> ''' + str(len(stops)) + '''</p>
    <p style="font-size:11px; color:gray; margin-bottom:0">
    Use o controle de camadas para mostrar/ocultar linhas específicas.<br>
    Clique nas linhas e paradas para mais informações.
    </p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # Salvar mapa
    print(f"\nSalvando mapa em: {output_html}")
    m.save(output_html)
    
    print(f"✓ Mapa salvo com sucesso!")
    print(f"\nAbra o arquivo '{output_html}' no navegador para visualizar.")
    
    # Retornar estatísticas
    return {
        'total_routes': len(routes),
        'unique_lines': len(unique_routes),
        'total_stops': len(stops),
        'output_file': output_html
    }


def create_simple_map(routes_file, output_html='bus_routes_simple.html'):
    """
    Cria mapa simples apenas com rotas (sem controle de camadas)
    Útil para visualização rápida de todas as linhas ao mesmo tempo
    """
    
    print("Criando mapa simples...")
    
    routes = gpd.read_file(routes_file)
    
    if routes.crs != 'EPSG:4326':
        routes = routes.to_crs('EPSG:4326')
    
    bounds = routes.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='CartoDB positron'
    )
    
    # Adicionar todas as rotas
    for idx, route in routes.iterrows():
        coords = [(coord[1], coord[0]) for coord in route.geometry.coords]
        
        folium.PolyLine(
            coords,
            color=route['route_color'],
            weight=2,
            opacity=0.6,
            popup=f"{route['route_short_name']} - {route['route_long_name']}",
            tooltip=route['route_short_name']
        ).add_to(m)
    
    plugins.Fullscreen().add_to(m)
    
    m.save(output_html)
    print(f"✓ Mapa simples salvo em: {output_html}")


def create_heatmap(stops_file, output_html='bus_stops_heatmap.html'):
    """
    Cria mapa de calor mostrando densidade de paradas
    """
    
    print("Criando mapa de calor...")
    
    stops = gpd.read_file(stops_file)
    
    if stops.crs != 'EPSG:4326':
        stops = stops.to_crs('EPSG:4326')
    
    # Preparar dados para heatmap
    heat_data = [[point.y, point.x] for point in stops.geometry]
    
    bounds = stops.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='CartoDB dark_matter'
    )
    
    plugins.HeatMap(
        heat_data,
        radius=15,
        blur=20,
        max_zoom=13
    ).add_to(m)
    
    plugins.Fullscreen().add_to(m)
    
    m.save(output_html)
    print(f"✓ Mapa de calor salvo em: {output_html}")


if __name__ == "__main__":
    import sys
    import os
    
    # Verificar se arquivos existem
    routes_file = 'bus_routes.geojson'
    stops_file = 'bus_stops.geojson'
    
    if not os.path.exists(routes_file):
        print(f"ERRO: Arquivo {routes_file} não encontrado!")
        print("Execute primeiro o script 'scrape_bus_routes_v2.py'")
        sys.exit(1)
    
    if not os.path.exists(stops_file):
        print(f"AVISO: Arquivo {stops_file} não encontrado!")
        print("Criando mapa apenas com rotas...")
        stops_file = None
    
    print("="*60)
    print("GERAÇÃO DE MAPAS HTML INTERATIVOS")
    print("="*60 + "\n")
    
    # Criar mapa completo interativo
    if stops_file and os.path.exists(stops_file):
        stats = create_interactive_map(
            routes_file, 
            stops_file, 
            'bus_routes_interactive.html'
        )
        print(f"\n✓ Mapa interativo criado com {stats['unique_lines']} linhas")
    
    # Criar mapa simples
    create_simple_map(routes_file, 'bus_routes_all.html')
    
    # Criar mapa de calor
    if stops_file and os.path.exists(stops_file):
        create_heatmap(stops_file, 'bus_stops_heatmap.html')
    
    print("\n" + "="*60)
    print("MAPAS CRIADOS COM SUCESSO!")
    print("="*60)
    print("\nArquivos gerados:")
    print("  1. bus_routes_interactive.html - Mapa completo com controle de camadas")
    print("  2. bus_routes_all.html - Todas as rotas visíveis ao mesmo tempo")
    print("  3. bus_stops_heatmap.html - Mapa de calor das paradas")
    print("\nAbra qualquer arquivo .html no navegador para visualizar.")

GERAÇÃO DE MAPAS HTML INTERATIVOS

Carregando dados...
✓ 636 rotas carregadas
✓ 14632 paradas carregadas

Criando mapa centrado em: -23.4544, -51.1200

Adicionando 140 linhas ao mapa...

Salvando mapa em: bus_routes_interactive.html
✓ Mapa salvo com sucesso!

Abra o arquivo 'bus_routes_interactive.html' no navegador para visualizar.

✓ Mapa interativo criado com 140 linhas
Criando mapa simples...
✓ Mapa simples salvo em: bus_routes_all.html
Criando mapa de calor...
✓ Mapa de calor salvo em: bus_stops_heatmap.html

MAPAS CRIADOS COM SUCESSO!

Arquivos gerados:
  1. bus_routes_interactive.html - Mapa completo com controle de camadas
  2. bus_routes_all.html - Todas as rotas visíveis ao mesmo tempo
  3. bus_stops_heatmap.html - Mapa de calor das paradas

Abra qualquer arquivo .html no navegador para visualizar.
